In [133]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt 
import seaborn as sns

from imblearn.over_sampling import SMOTE 
from sklearn.preprocessing import LabelEncoder 
from sklearn.model_selection import train_test_split 
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report,confusion_matrix, cohen_kappa_score, roc_auc_score
from sklearn.metrics import precision_score, recall_score,f1_score
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier

In [ ]:
# Import libraries
import pandas as pd

# Load dataset
loan_df = pd.read_csv("Loan.csv")

# (i) Number of rows and columns
rows, cols = loan_df.shape
print(f"Number of rows: {rows}, Number of columns: {cols}")

# (ii) Data types of all variables
print("\nData Types:")
print(loan_df.dtypes)

# (iii) Five-point summary for numerical variables
print("\nFive-point Summary (Numerical):")
print(loan_df.describe())

# (iv) Summary of categorical variables
categorical_cols = loan_df.select_dtypes(include='object').columns
for col in categorical_cols:
    print(f"\nColumn: {col}")
    print(loan_df[col].value_counts())
    print(loan_df[col].value_counts(normalize=True)*100)


loan_df.describe(include='object')

df.head()

import matplotlib.pyplot as plt
import seaborn as sns

# (i) Missing values
print("Missing Values per Column:")
print(loan_df.isnull().sum())

# Visualize missing values
sns.heatmap(loan_df.isnull(), cbar=False, yticklabels=False)
plt.title("Missing Values Heatmap")
plt.show()

# Impute missing values (if any)
loan_df['Balance'].fillna(loan_df['Balance'].median(), inplace=True)
loan_df['EstimatedSalary'].fillna(loan_df['EstimatedSalary'].median(), inplace=True)

# (ii) Outliers in numerical variables
num_cols = ['CreditScore', 'Age', 'Balance', 'EstimatedSalary']
for col in num_cols:
    plt.figure(figsize=(6,3))
    sns.boxplot(loan_df[col])
    plt.title(f"Boxplot of {col}")
    plt.show()

# Remove outliers using IQR
for col in num_cols:
    Q1 = loan_df[col].quantile(0.25)
    Q3 = loan_df[col].quantile(0.75)
    IQR = Q3 - Q1
    lower = Q1 - 1.5*IQR
    upper = Q3 + 1.5*IQR
    loan_df = loan_df[(loan_df[col] >= lower) & (loan_df[col] <= upper)]

# (iii) Target distribution
sns.countplot(x='Loan', data=loan_df)
plt.title("Loan Target Distribution")
plt.show()
print(loan_df['Loan'].value_counts(normalize=True)*100)

# (iv) Encode textual categorical variables
loan_df['Gender'] = loan_df['Gender'].map({'Male':0,'Female':1})
loan_df = pd.get_dummies(loan_df, columns=['Geography'], drop_first=True)


df.head()

# Correlation matrix
plt.figure(figsize=(10,6))
sns.heatmap(loan_df.corr(), annot=True, cmap='coolwarm')
plt.title("Correlation Matrix")
plt.show()

# Categorical vs Target plots
plt.figure(figsize=(6,4))
sns.barplot(x='HasCrCard', y='Loan', data=loan_df)
plt.title("Loan vs HasCrCard")
plt.show()

plt.figure(figsize=(6,4))
sns.barplot(x='IsActiveMember', y='Loan', data=loan_df)
plt.title("Loan vs IsActiveMember")
plt.show()

# Numerical vs Target plots
num_cols = ['CreditScore', 'Age', 'Balance', 'EstimatedSalary']
for col in num_cols:
    plt.figure(figsize=(6,4))
    sns.boxplot(x='Loan', y=col, data=loan_df)
    plt.title(f"{col} vs Loan")
    plt.show()

# Correlation with target
print("Correlation with Loan:")
print(loan_df.corr()['Loan'].sort_values(ascending=False))


loan_df['Age'].fillna(loan_df['Age'].median(),inplace=True)
# Clean Gender column
loan_df['Gender'].fillna(loan_df['Gender'].mode()[0],inplace=True)

loan_df[num_cols] = loan_df[num_cols].astype('float64')

loan_df.info()

from sklearn.model_selection import train_test_split

X = loan_df.drop(['CustomerId','Loan'], axis=1)
y = loan_df['Loan']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42, stratify=y)
print(f"Train size: {X_train.shape}, Test size: {X_test.shape}")


from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, cohen_kappa_score, classification_report

# Base model
base_model = LogisticRegression(max_iter=500, random_state=42)
base_model.fit(X_train, y_train)
y_pred = base_model.predict(X_test)

# Evaluation metrics
accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred)
recall = recall_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)
kappa = cohen_kappa_score(y_test, y_pred)

print(f"Accuracy: {accuracy:.2f}")
print(f"Precision: {precision:.2f}")
print(f"Recall: {recall:.2f}")
print(f"F1 Score: {f1:.2f}")
print(f"Cohen's Kappa: {kappa:.2f}")

# Significant coefficients
coeff_df = pd.DataFrame({'Feature': X_train.columns, 'Coefficient': base_model.coef_[0]}).sort_values(by='Coefficient', ascending=False)
print(coeff_df)



# Compute VIF
import statsmodels.api as sm
X_vif = X_train.astype(float)
X_vif = sm.add_constant(X_vif)
vif_df = pd.DataFrame()
vif_df['Feature'] = X_vif.columns
vif_df['VIF'] = [variance_inflation_factor(X_vif.values, i) for i in range(X_vif.shape[1])]
print(vif_df)


from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegressionCV
from sklearn.metrics import confusion_matrix, classification_report, roc_auc_score

# Scaling numerical features
num_cols = ['CreditScore', 'Age', 'Balance', 'EstimatedSalary', 'Tenure']
scaler = StandardScaler()
X_train[num_cols] = scaler.fit_transform(X_train[num_cols])
X_test[num_cols] = scaler.transform(X_test[num_cols])

# Logistic Regression with Cross Validation and class weight
final_model = LogisticRegressionCV(
    cv=5, penalty='l2', solver='liblinear', scoring='f1', max_iter=500, class_weight='balanced', random_state=42
)
final_model.fit(X_train, y_train)
y_pred_final = final_model.predict(X_test)
y_proba_final = final_model.predict_proba(X_test)[:,1]

# Evaluation
accuracy = accuracy_score(y_test, y_pred_final)
precision = precision_score(y_test, y_pred_final)
recall = recall_score(y_test, y_pred_final)
f1 = f1_score(y_test, y_pred_final)
kappa = cohen_kappa_score(y_test, y_pred_final)
roc_auc = roc_auc_score(y_test, y_proba_final)

print(f"Accuracy: {accuracy:.3f}")
print(f"Precision: {precision:.3f}")
print(f"Recall: {recall:.3f}")
print(f"F1 Score: {f1:.3f}")
print(f"Cohen's Kappa: {kappa:.3f}")
print(f"ROC AUC: {roc_auc:.3f}")



coeff_df = pd.DataFrame({'Feature': X_train.columns, 'Coefficient': final_model.coef_[0]}).sort_values(by='Coefficient', ascending=False)
print(coeff_df)

# Optional: Plot top 5 features
import matplotlib.pyplot as plt
top_features = coeff_df.head(5)
plt.barh(top_features['Feature'], top_features['Coefficient'])
plt.xlabel("Coefficient Value")
plt.ylabel("Feature")
plt.title("Top 5 Features Affecting Loan Repayment")
plt.show()


weighted_model = LogisticRegression(
    class_weight='balanced',  # automatically adjusts weights
    max_iter=1000
)

weighted_model.fit(X_train, y_train)

y_pred_weighted = weighted_model.predict(X_test)

print("=== Class Weight ===")
print(classification_report(y_test, y_pred_weighted))
print(accuracy_score(y_test, y_pred_weighted))
print("ROC-AUC:", roc_auc_score(y_test, weighted_model.predict_proba(X_test)[:, 1]))

from imblearn.over_sampling import SMOTE

smote = SMOTE(random_state=42)

# Apply ONLY on training data
X_train_smote, y_train_smote = smote.fit_resample(X_train, y_train)

print("Before SMOTE:", np.bincount(y_train))
print("After SMOTE:", np.bincount(y_train_smote))

smote_model = LogisticRegression(max_iter=1000)
smote_model.fit(X_train_smote, y_train_smote)

y_pred_smote = smote_model.predict(X_test)

print("=== SMOTE ===")
print(classification_report(y_test, y_pred_smote))
print(accuracy_score(y_test, y_pred_smote))
print("ROC-AUC:", roc_auc_score(y_test, smote_model.predict_proba(X_test)[:, 1]))


# --- 1️⃣ SMOTE pipeline ---
pipeline_smote = Pipeline([
    ('smote', SMOTE(random_state=42)),
    ('model', LogisticRegression(max_iter=1000))
])

scores_smote = cross_val_score(
    pipeline_smote, X, y, cv=5, scoring='accuracy'
)

# --- 2️⃣ Class Weight Logistic Regression ---
weighted_model = LogisticRegression(
    class_weight='balanced', max_iter=1000
)

scores_weighted = cross_val_score(
    weighted_model, X, y, cv=5, scoring='accuracy'
)

# --- 3️⃣ LogisticRegressionCV with class_weight ---
logreg_cv_model = LogisticRegressionCV(
    cv=5,
    class_weight='balanced',   # handles imbalance
    max_iter=1000,
    scoring='accuracy',
    n_jobs=-1
)

scores_logregcv = cross_val_score(
    logreg_cv_model, X, y, cv=5, scoring='accuracy'
)

# --- 4️⃣ Print comparison ---
print("SMOTE CV Accuracy:", np.mean(scores_smote))
print("Class Weight CV Accuracy:", np.mean(scores_weighted))
print("LogisticRegressionCV Accuracy:", np.mean(scores_logregcv))

from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, cohen_kappa_score, classification_report

# Base model
base_model = LogisticRegression(max_iter=500, random_state=42)
base_model.fit(X_train, y_train)
y_pred = base_model.predict(X_test)

# Evaluation metrics
accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred)
recall = recall_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)
kappa = cohen_kappa_score(y_test, y_pred)

print(f"Accuracy: {accuracy:.2f}")
print(f"Precision: {precision:.2f}")
print(f"Recall: {recall:.2f}")
print(f"F1 Score: {f1:.2f}")
print(f"Cohen's Kappa: {kappa:.2f}")

# Significant coefficients
coeff_df = pd.DataFrame({'Feature': X_train.columns, 'Coefficient': base_model.coef_[0]}).sort_values(by='Coefficient', ascending=False)
print(coeff_df)


from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegressionCV
from sklearn.metrics import confusion_matrix, classification_report, roc_auc_score

# Scaling numerical features
num_cols = ['CreditScore', 'Age', 'Balance', 'EstimatedSalary', 'Tenure']
scaler = StandardScaler()
X_train[num_cols] = scaler.fit_transform(X_train[num_cols])
X_test[num_cols] = scaler.transform(X_test[num_cols])

# Logistic Regression with Cross Validation and class weight
final_model = LogisticRegressionCV(
    cv=5, penalty='l2', solver='liblinear', scoring='f1', max_iter=500, class_weight='balanced', random_state=42
)
final_model.fit(X_train, y_train)
y_pred_final = final_model.predict(X_test)
y_proba_final = final_model.predict_proba(X_test)[:,1]

# Evaluation
accuracy = accuracy_score(y_test, y_pred_final)
precision = precision_score(y_test, y_pred_final)
recall = recall_score(y_test, y_pred_final)
f1 = f1_score(y_test, y_pred_final)
kappa = cohen_kappa_score(y_test, y_pred_final)
roc_auc = roc_auc_score(y_test, y_proba_final)

print(f"Accuracy: {accuracy:.3f}")
print(f"Precision: {precision:.3f}")
print(f"Recall: {recall:.3f}")
print(f"F1 Score: {f1:.3f}")
print(f"Cohen's Kappa: {kappa:.3f}")
print(f"ROC AUC: {roc_auc:.3f}")


smote = SMOTE(random_state = 42)
X_train_sm , y_train_sm = smote.fit_resample(X_train,y_train)
print("Before SMOTE:", y_train.value_counts())
print("After SMOTE:", y_train_sm.value_counts())

# Import models
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from xgboost import XGBClassifier


# Metrics
from sklearn.metrics import accuracy_score, precision_score, recall_score
from sklearn.metrics import f1_score, roc_auc_score

import pandas as pd
# ----------------------------
# Define Models
# ----------------------------
models = {
    "KNN": KNeighborsClassifier(),
    "Naive Bayes": GaussianNB(),
    "Decision Tree": DecisionTreeClassifier(random_state=42),
    "Random Forest": RandomForestClassifier(random_state=42),
    "Gradient Boosting": GradientBoostingClassifier(random_state=42),
    "XGBoost": XGBClassifier(use_label_encoder=False, eval_metric='logloss', random_state=42)
}


# ----------------------------
# Store Results
# ----------------------------
results = []
# ----------------------------
# Train and Evaluate Models
# ----------------------------
for name, model in models.items():
    
    # Train model on SMOTE data
    model.fit(X_train_sm, y_train_sm)
    
    # Predictions
    y_pred = model.predict(X_test_sm)
    y_prob = model.predict_proba(X_test_sm)[:,1]
    
    # Metrics
    acc = accuracy_score(y_test, y_pred)
    prec = precision_score(y_test, y_pred)
    rec = recall_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred)
    roc = roc_auc_score(y_test, y_prob)
    
    results.append([name, acc, prec, rec, f1, roc])

# ----------------------------
# Create Results Table
# ----------------------------
results_df = pd.DataFrame(results, 
                          columns=["Model", "Accuracy", "Precision", 
                                   "Recall", "F1 Score", "ROC-AUC"])

# Sort by Recall (since this is important for loan default)
results_df = results_df.sort_values(by="Recall", ascending=False)

print(results_df)

models = {
    "KNN": KNeighborsClassifier(),
    "Naive Bayes": GaussianNB(),
    "Decision Tree": DecisionTreeClassifier(random_state=42),
    "Random Forest": RandomForestClassifier(random_state=42),
    "Gradient Boosting": GradientBoostingClassifier(random_state=42),
    "XGBoost": XGBClassifier(use_label_encoder=False, eval_metric='logloss', random_state=42)
}


# ----------------------------
# Train and Evaluate Models on SMOTE
# ----------------------------
results = []

for name, model in models.items():
    
    # Train model on SMOTE data
    model.fit(X_train_sm, y_train_sm)
    
    # Predict on original test data
    y_pred = model.predict(X_test)          # use original X_test
    y_prob = model.predict_proba(X_test)[:,1]  # probability for ROC-AUC
    
    # Compute metrics
    acc = accuracy_score(y_test, y_pred)
    prec = precision_score(y_test, y_pred)
    rec = recall_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred)
    roc = roc_auc_score(y_test, y_prob)
    
    # Store results
    results.append([name, acc, prec, rec, f1, roc])

# ----------------------------
# Create Results Table
# ----------------------------
results_df = pd.DataFrame(
    results, 
    columns=["Model", "Accuracy", "Precision", "Recall", "F1 Score", "ROC-AUC"]
)

# Sort by Recall (important for loan default)
results_df = results_df.sort_values(by="Recall", ascending=False)

print(results_df)